To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

You can now train embedding models 1.8-3.3x faster with 20% less VRAM. [Blog](https://unsloth.ai/docs/new/embedding-finetuning)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

3x faster LLM training with 30% less VRAM and 500K context. [3x faster](https://unsloth.ai/docs/new/3x-faster-training-packing) • [500K Context](https://unsloth.ai/docs/new/500k-context-length-fine-tuning)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

### Unsloth

Let's prepare the OCR model to our local first

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR-2", local_dir = "deepseek_ocr2")

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import AutoModel
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit", # Qwen 3 vision support
    "unsloth/Qwen3-VL-8B-Thinking-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Thinking-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr2",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    auto_model = AutoModel,
    trust_remote_code = True,
    unsloth_force_compile = True,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

### Baseline DeepSeek-OCR Output On A FineTree Sample


In [ ]:
from datasets import load_dataset
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
import platform
import re
import socket
import subprocess
from zoneinfo import ZoneInfo

TRAIN_DATASET_ID = "asafd60/FineTree-value-path-with-meta-no-bbox-train-approved"
VAL_DATASET_ID = "asafd60/FineTree-value-path-with-meta-no-bbox-validation-approved"
DATASET_SPLIT = "train"
IMAGE_COLUMN = "image"
TEXT_COLUMN = "text"
OCR_PROMPT = "<image>\nFree OCR. "
OCR_BASE_SIZE = 1280
OCR_IMAGE_SIZE = 1280
OCR_CROP_MODE = False

RUN_ID = f"deepseek-ocr-finetree-{time.strftime('%Y%m%d-%H%M%S')}"
RUN_OUTPUT_DIR = Path("outputs_finetree_deepseekocr") / RUN_ID
BENCHMARK_BUNDLE_DIR = RUN_OUTPUT_DIR / "benchmark_submission"
PREDICTIONS_DIR = BENCHMARK_BUNDLE_DIR / "predictions"
RAW_PREDICTIONS_PATH = BENCHMARK_BUNDLE_DIR / "raw_predictions.jsonl"
BUNDLE_ARGS_PATH = BENCHMARK_BUNDLE_DIR / "args.json"
BUNDLE_LOGGING_PATH = BENCHMARK_BUNDLE_DIR / "logging.jsonl"
FINAL_ADAPTER_DIR = RUN_OUTPUT_DIR / "deepseek_ocr_lora_finetree_v2"
MERGED_16BIT_DIR = RUN_OUTPUT_DIR / "deepseek_ocr_finetree_merged_16bit"
BENCHMARK_EVAL_DIR = RUN_OUTPUT_DIR / "benchmark_eval"
BENCHMARK_IMAGE_DIR = BENCHMARK_EVAL_DIR / "images"
RUN_ARCHIVE_BASE = RUN_OUTPUT_DIR / f"{RUN_ID}_benchmark_submission"

RUN_OUTPUT_DIR.mkdir(parents = True, exist_ok = True)
BENCHMARK_BUNDLE_DIR.mkdir(parents = True, exist_ok = True)
PREDICTIONS_DIR.mkdir(parents = True, exist_ok = True)
BENCHMARK_IMAGE_DIR.mkdir(parents = True, exist_ok = True)

BENCHMARK_METADATA_FIELDS = [
    "checkpoint_name", "model", "tuner_type", "dataset", "validation_dataset", "use_hf",
    "freeze_vit", "vit_lr", "freeze_aligner", "enable_thinking", "add_non_thinking_prefix",
    "torch_dtype", "num_train_epochs", "per_device_train_batch_size", "per_device_eval_batch_size",
    "learning_rate", "lora_rank", "lora_alpha", "target_modules", "gradient_accumulation_steps",
    "output_dir", "eval_steps", "save_steps", "save_total_limit", "logging_steps", "max_length",
    "warmup_ratio", "dataset_num_proc", "dataloader_num_workers", "eval_on_start", "max_pixels",
    "temperature", "gradient_checkpointing", "gradient_checkpointing_kwargs", "truncation_strategy",
    "CUDA_VISIBLE_DEVICES", "MAX_PIXELS", "gpu_used", "torch_env_used", "platform",
]

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def israel_now_iso():
    return datetime.now(ZoneInfo("Asia/Jerusalem")).isoformat()

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents = True, exist_ok = True)
    path.write_text(json.dumps(payload, ensure_ascii = False, indent = 2) + "\n", encoding = "utf-8")
    return path

def collect_environment_snapshot():
    gpu_names = []
    gpu_count = 0
    torch_version = None
    cuda_runtime_version = None
    try:
        import torch
        torch_version = getattr(torch, "__version__", None)
        cuda_runtime_version = getattr(getattr(torch, "version", None), "cuda", None)
        if torch.cuda.is_available():
            gpu_count = int(torch.cuda.device_count())
            gpu_names = [str(torch.cuda.get_device_name(i)) for i in range(gpu_count)]
    except Exception:
        pass
    nvidia_smi = None
    try:
        completed = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            check = False,
            capture_output = True,
            text = True,
        )
        if completed.returncode == 0:
            nvidia_smi = completed.stdout.strip() or None
            if not gpu_names and nvidia_smi:
                gpu_names = [line.split(",", 1)[0].strip() for line in nvidia_smi.splitlines() if line.strip()]
                gpu_count = len(gpu_names)
    except Exception:
        pass
    return {
        "platform": platform.platform(),
        "platform_machine": platform.machine(),
        "python_version": platform.python_version(),
        "hostname": socket.gethostname(),
        "torch_version": torch_version,
        "cuda_runtime_version": cuda_runtime_version,
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "max_pixels_env": os.environ.get("MAX_PIXELS"),
        "gpu_count": gpu_count,
        "gpu_names": gpu_names,
        "gpu_used": " | ".join(gpu_names) if gpu_names else None,
        "nvidia_smi": nvidia_smi,
    }

def _parse_step_progress(row):
    text = str(row.get("global_step/max_steps") or "").strip()
    if "/" not in text:
        return None, None
    left, right = text.split("/", 1)
    try:
        return int(left), int(right)
    except ValueError:
        return None, None

def _checkpoint_step_map(output_dir):
    step_map = {}
    for path in Path(output_dir).iterdir():
        if not path.is_dir():
            continue
        match = re.fullmatch(r"checkpoint-(\d+)", path.name)
        if match is None:
            continue
        step_map[int(match.group(1))] = path
    return step_map

def select_best_checkpoint(output_dir, logging_rows):
    step_map = _checkpoint_step_map(output_dir)
    eval_candidates = []
    for row in logging_rows:
        if "eval_loss" not in row:
            continue
        try:
            eval_loss = float(row["eval_loss"])
        except Exception:
            continue
        global_step, max_steps = _parse_step_progress(row)
        if global_step is None or global_step not in step_map:
            continue
        epoch = row.get("epoch")
        epoch_value = float(epoch) if isinstance(epoch, (int, float)) else None
        eval_candidates.append({
            "checkpoint_name": step_map[global_step].name,
            "checkpoint_path": str(step_map[global_step]),
            "epoch": epoch_value,
            "global_step": global_step,
            "max_steps": max_steps,
            "eval_loss": eval_loss,
            "selection_metric": "eval_loss",
            "selection_reason": "lowest_eval_loss",
            "adapter_only": True,
        })
    if eval_candidates:
        return sorted(
            eval_candidates,
            key = lambda item: (
                float(item["eval_loss"]),
                -(item["epoch"] if item["epoch"] is not None else -1.0),
                -int(item["global_step"]),
            ),
        )[0]
    if step_map:
        final_step = max(step_map)
        return {
            "checkpoint_name": step_map[final_step].name,
            "checkpoint_path": str(step_map[final_step]),
            "epoch": None,
            "global_step": final_step,
            "max_steps": None,
            "eval_loss": None,
            "selection_metric": "fallback",
            "selection_reason": "final_checkpoint",
            "adapter_only": True,
        }
    raise FileNotFoundError(f"No checkpoint-* directories found under {output_dir}")

def _maybe_parse_json_string(text):
    if not isinstance(text, str):
        return None
    stripped = text.strip()
    if not stripped.startswith("{"):
        return None
    try:
        payload = json.loads(stripped)
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None

def _maybe_extract_json_object(text):
    parsed = _maybe_parse_json_string(text)
    if parsed is not None:
        return parsed
    if not isinstance(text, str):
        return None
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end <= start:
        return None
    try:
        payload = json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None

def extract_prediction_payload(row):
    if isinstance(row, dict):
        if isinstance(row.get("pages"), list) or isinstance(row.get("facts"), list):
            return row
        for key in ("prediction_json", "parsed_prediction", "prediction", "response", "generated_text", "output", "text"):
            value = row.get(key)
            if isinstance(value, dict) and (isinstance(value.get("pages"), list) or isinstance(value.get("facts"), list)):
                return value
            parsed = _maybe_extract_json_object(value)
            if parsed is not None:
                return parsed
    if isinstance(row, str):
        parsed = _maybe_extract_json_object(row)
        if parsed is not None:
            return parsed
    raise ValueError(
        "Could not extract a benchmark JSON payload from inference output row. "
        "Refusing to fall back to labels or other non-prediction fields."
    )

def materialize_prediction_json_files(result_jsonl_path, output_dir, prefix = "pred_"):
    result_path = Path(result_jsonl_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents = True, exist_ok = True)
    written_paths = []
    for index, line in enumerate(result_path.read_text(encoding = "utf-8").splitlines(), start = 1):
        if not line.strip():
            continue
        payload = json.loads(line)
        prediction = extract_prediction_payload(payload)
        path = output_dir / f"{prefix}{index:04d}.json"
        path.write_text(json.dumps(prediction, ensure_ascii = False, indent = 2) + "\n", encoding = "utf-8")
        written_paths.append(path)
    if not written_paths:
        raise ValueError(f"No prediction rows were written from {result_jsonl_path}")
    return written_paths

def build_benchmark_model_metadata(training_args, environment, checkpoint_name):
    metadata_out = {field_name: None for field_name in BENCHMARK_METADATA_FIELDS}
    metadata_out.update({str(key): value for key, value in training_args.items() if str(key) in metadata_out})
    metadata_out["checkpoint_name"] = checkpoint_name
    metadata_out["validation_dataset"] = training_args.get("validation_dataset") or training_args.get("val_dataset")
    metadata_out["CUDA_VISIBLE_DEVICES"] = training_args.get("CUDA_VISIBLE_DEVICES") or environment.get("cuda_visible_devices")
    metadata_out["MAX_PIXELS"] = training_args.get("MAX_PIXELS") or environment.get("max_pixels_env")
    metadata_out["gpu_used"] = environment.get("gpu_used")
    metadata_out["torch_env_used"] = " | ".join(
        str(value)
        for value in (
            f"torch {environment.get('torch_version')}" if environment.get("torch_version") else None,
            f"cuda {environment.get('cuda_runtime_version')}" if environment.get("cuda_runtime_version") else None,
            f"python {environment.get('python_version')}" if environment.get("python_version") else None,
        )
        if value
    ) or None
    metadata_out["platform"] = environment.get("platform")
    return metadata_out

def build_submission_info_payload(model_metadata, training_args, environment, run, selected_checkpoint, artifacts):
    return {
        "model_metadata": {str(key): value for key, value in model_metadata.items()},
        "training_args": {str(key): value for key, value in training_args.items()},
        "environment": {str(key): value for key, value in environment.items()},
        "run": {str(key): value for key, value in run.items()},
        "selected_checkpoint": {str(key): value for key, value in selected_checkpoint.items()},
        "artifacts": {str(key): value for key, value in artifacts.items()},
    }

train_dataset_raw = load_dataset(TRAIN_DATASET_ID, split = DATASET_SPLIT)
val_dataset_raw = load_dataset(VAL_DATASET_ID, split = DATASET_SPLIT)

required_columns = {IMAGE_COLUMN, TEXT_COLUMN}
for dataset_name, dataset_obj in (("train", train_dataset_raw), ("validation", val_dataset_raw)):
    missing_columns = required_columns - set(dataset_obj.column_names)
    if missing_columns:
        raise ValueError(f"{dataset_name} dataset is missing required columns: {sorted(missing_columns)}")

def has_non_empty_facts(sample):
    compact = str(sample[TEXT_COLUMN]).replace(" ", "")
    return "\"facts\":[]" not in compact

EVAL_SAMPLE_INDEX = next((idx for idx, sample in enumerate(val_dataset_raw) if has_non_empty_facts(sample)), 0)

print(train_dataset_raw)
print(val_dataset_raw)
print(train_dataset_raw.features)
print(f"Ignoring prompt columns: {[c for c in train_dataset_raw.column_names if c not in (IMAGE_COLUMN, TEXT_COLUMN)]}")
print(f"Preview validation sample index: {EVAL_SAMPLE_INDEX}")
print(f"Run output dir: {RUN_OUTPUT_DIR}")


In [ ]:
# Save one FineTree validation image for quick before / after inspection
eval_sample = val_dataset_raw[EVAL_SAMPLE_INDEX]
eval_sample[IMAGE_COLUMN].save("your_image.jpg")


In [ ]:
eval_sample[IMAGE_COLUMN]


In [ ]:
# prompt = OCR_PROMPT
prompt = OCR_PROMPT
image_file = "your_image.jpg"
output_path = "your/output/dir"
# infer(self, tokenizer, prompt = "", image_file = "", output_path = " ", base_size = OCR_BASE_SIZE, image_size = OCR_IMAGE_SIZE, crop_mode = OCR_CROP_MODE, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 768, image_size = 768, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# This notebook uses the Large preset everywhere.

res = model.infer(tokenizer, prompt = prompt, image_file = image_file, output_path = output_path, base_size = OCR_BASE_SIZE, image_size = OCR_IMAGE_SIZE, crop_mode = OCR_CROP_MODE, save_results = True, test_compress = False)


In [ ]:
eval_sample[TEXT_COLUMN]


<h3>Use this sample as the baseline output before finetuning on FineTree.</h3>


# Let's finetune Deepseek-OCR !

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

<a name="Data"></a>
### Data Prep
We train on the FineTree approved no-bbox split with page meta plus fact `value` and `path`, and evaluate plus benchmark on the matching approved validation split.
As requested, the notebook ignores the dataset `system` and `instruction` columns and trains directly from `image -> text` with the fixed `Free OCR` prompt.
Your `text` column is structured JSON, so this setup trains DeepSeek-OCR to emit that reduced JSON target while using the plain OCR-style prompt.

Train dataset: [asafd60/FineTree-value-path-with-meta-no-bbox-train-approved](https://huggingface.co/datasets/asafd60/FineTree-value-path-with-meta-no-bbox-train-approved)
Validation dataset: [asafd60/FineTree-value-path-with-meta-no-bbox-validation-approved](https://huggingface.co/datasets/asafd60/FineTree-value-path-with-meta-no-bbox-validation-approved)


To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "<|User|>",
  "content": "",
  "images": []
},
{ "role": "<|Assistant|>",
  "content": ""
},
]
```

In [ ]:
instruction = OCR_PROMPT

def convert_to_conversation(sample):
    """Convert FineTree rows into the DeepSeek-OCR conversation format."""
    conversation = [
        {
            "role": "<|User|>",
            "content": instruction,
            "images": [sample[IMAGE_COLUMN]],
        },
        {
            "role": "<|Assistant|>",
            "content": sample[TEXT_COLUMN],
        },
    ]
    return {"messages": conversation}

print(f"Training rows: {len(train_dataset_raw)}")
print(f"Validation rows: {len(val_dataset_raw)}")
print(f"Extra columns ignored during training: {[c for c in train_dataset_raw.column_names if c not in (IMAGE_COLUMN, TEXT_COLUMN)]}")


Let's convert the dataset into the "correct" format for finetuning:

In [ ]:
converted_train_dataset = [convert_to_conversation(sample) for sample in train_dataset_raw]
converted_eval_dataset = [convert_to_conversation(sample) for sample in val_dataset_raw]


We look at how the conversations are structured for the first example:

In [ ]:
converted_train_dataset[0]


In [ ]:
# @title Create datacollator

import torch
import math
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
import io

from deepseek_ocr2.modeling_deepseekocr2 import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

@dataclass
class DeepSeekOCR2DataCollator:
    """
    Args:
        tokenizer: Tokenizer
        model: Model
        image_size: Size for image patches (default: 768)
        base_size: Size for global view (default: 1024)
        crop_mode: Whether to use dynamic cropping for large images
        train_on_responses_only: If True, only train on assistant responses (mask user prompts)
    """
    tokenizer: Any
    model: Any
    image_size: int = 768
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True
    fail_on_sequence_overflow: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 768,
        base_size: int = 1024,
        crop_mode: bool = True,
        train_on_responses_only: bool = True,
        fail_on_sequence_overflow: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype  # Get dtype from model
        self.train_on_responses_only = train_on_responses_only
        self.fail_on_sequence_overflow = fail_on_sequence_overflow

        self.image_transform = BasicImageTransform(
            mean = (0.5, 0.5, 0.5),
            std = (0.5, 0.5, 0.5),
            normalize = True
        )
        self.patch_size = 16
        self.downsample_ratio = 4

        # Get BOS token ID from tokenizer
        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0
            print(f"Warning: tokenizer has no bos_token_id, using default: {self.bos_id}")

    def deserialize_image(self, image_data) -> Image.Image:
        """Convert image data (bytes dict or PIL Image) to PIL Image in RGB mode"""
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            image_bytes = image_data['bytes']
            image = Image.open(io.BytesIO(image_bytes))
            return image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")

    def calculate_image_token_count(self, image: Image.Image, crop_ratio: Tuple[int, int]) -> int:
        """Calculate the number of tokens this image will generate"""
        num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

        width_crop_num, height_crop_num = crop_ratio

        if self.crop_mode:
            img_tokens = num_queries_base * num_queries_base + 1
            if width_crop_num > 1 or height_crop_num > 1:
                img_tokens += (num_queries * width_crop_num) * (num_queries * height_crop_num)
        else:
            img_tokens = num_queries * num_queries + 1

        return img_tokens

    def process_image(self, image: Image.Image) -> Tuple[List, List, List, List, Tuple[int, int]]:
        """
        Process a single image based on crop_mode and size thresholds

        Returns:
            Tuple of (images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio)
        """
        images_list = []
        images_crop_list = []
        images_spatial_crop = []

        if self.crop_mode:
            # Determine crop ratio based on image size
            if image.size[0] <= 768 and image.size[1] <= 768:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num = 2, max_num = 6,
                    image_size = self.image_size, use_thumbnail = False
                )

            # Process global view with padding
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color = tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))

            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            # Process local views (crops) if applicable
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(
                        self.image_transform(crop_img).to(self.dtype)
                    )

            # Calculate image tokens
            num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

            tokenized_image = ([self.image_token_id] * num_queries_base) * num_queries_base
            tokenized_image += [self.image_token_id]

            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num)) * (
                    num_queries * height_crop_num)

        else:  # crop_mode = False
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])

            # For smaller base sizes, resize; for larger, pad
            if self.base_size <= 768:
                resized_image = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(resized_image).to(self.dtype))
            else:
                global_view = ImageOps.pad(
                    image, (self.base_size, self.base_size),
                    color = tuple(int(x * 255) for x in self.image_transform.mean)
                )
                images_list.append(self.image_transform(global_view).to(self.dtype))

            num_queries = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries) * num_queries
            tokenized_image += [self.image_token_id]

        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages: List[Dict]) -> Dict[str, Any]:
        """
        Process a single conversation into model inputs.
        """

        # --- 1. Setup ---
        images = []
        for message in messages:
            if "images" in message and message["images"]:
                for img_data in message["images"]:
                    if img_data is not None:
                        pil_image = self.deserialize_image(img_data)
                        images.append(pil_image)

        if not images:
            raise ValueError("No images found in sample. Please ensure all samples contain images.")

        tokenized_str = []
        images_seq_mask = []
        images_list, images_crop_list, images_spatial_crop = [], [], []

        prompt_token_count = -1 # Index to start training
        assistant_started = False
        image_idx = 0

        # Add BOS token at the very beginning
        tokenized_str.append(self.bos_id)
        images_seq_mask.append(False)

        for message in messages:
            role = message["role"]
            content = message["content"]

            # Check if this is the assistant's turn
            if role == "<|Assistant|>":
                if not assistant_started:
                    # This is the split point. All tokens added *so far*
                    # are part of the prompt.
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True

                # Append the EOS token string to the *end* of assistant content
                content = f"{content.strip()} {self.tokenizer.eos_token}"

            # Split this message's content by the image token
            text_splits = content.split('<image>')

            for i, text_sep in enumerate(text_splits):
                # Tokenize the text part
                tokenized_sep = text_encode(self.tokenizer, text_sep, bos = False, eos = False)
                tokenized_str.extend(tokenized_sep)
                images_seq_mask.extend([False] * len(tokenized_sep))

                # If this text is followed by an <image> tag
                if i < len(text_splits) - 1:
                    if image_idx >= len(images):
                        raise ValueError(
                            f"Data mismatch: Found '<image>' token but no corresponding image."
                        )

                    # Process the image
                    image = images[image_idx]
                    img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)

                    images_list.extend(img_list)
                    images_crop_list.extend(crop_list)
                    images_spatial_crop.extend(spatial_crop)

                    # Add image placeholder tokens
                    tokenized_str.extend(tok_img)
                    images_seq_mask.extend([True] * len(tok_img))

                    image_idx += 1 # Move to the next image

        # --- 3. Validation and Final Prep ---
        if image_idx != len(images):
            raise ValueError(
                f"Data mismatch: Found {len(images)} images but only {image_idx} '<image>' tokens were used."
            )

        # If we never found an assistant message, we're in a weird state
        # (e.g., user-only prompt). We mask everything.
        if not assistant_started:
            print("Warning: No assistant message found in sample. Masking all tokens.")
            prompt_token_count = len(tokenized_str)

        if self.fail_on_sequence_overflow:
            tokenizer_max_length = getattr(self.tokenizer, "model_max_length", None)
            if isinstance(tokenizer_max_length, int) and 0 < tokenizer_max_length < 10**8:
                if len(tokenized_str) > tokenizer_max_length:
                    raise ValueError(
                        f"Sequence length {len(tokenized_str)} exceeds tokenizer.model_max_length={tokenizer_max_length}. "
                        "This notebook is configured to fail instead of truncating."
                    )

        # Prepare image tensors
        images_ori = torch.stack(images_list, dim = 0)
        images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype = torch.long)

        if images_crop_list:
            images_crop = torch.stack(images_crop_list, dim = 0)
        else:
            images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype = self.dtype)

        return {
            "input_ids": torch.tensor(tokenized_str, dtype = torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype = torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": images_spatial_crop_tensor,
            "prompt_token_count": prompt_token_count, # This is now accurate
        }

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """Collate batch of samples"""
        batch_data = []

        # Process each sample
        for feature in features:
            try:
                processed = self.process_single_sample(feature['messages'])
                batch_data.append(processed)
            except Exception as e:
                print(f"Error processing sample: {e}")
                continue

        if not batch_data:
            raise ValueError("No valid samples in batch")

        # Extract lists
        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]

        # Pad sequences
        input_ids = pad_sequence(input_ids_list, batch_first = True, padding_value = self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first = True, padding_value = False)

        # Create labels
        labels = input_ids.clone()

        # Mask padding tokens
        labels[labels == self.tokenizer.pad_token_id] = -100

        # Mask image tokens (model shouldn't predict these)
        labels[images_seq_mask] = -100

        # Mask user prompt tokens when train_on_responses_only = True (only train on assistant responses)
        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        # Create attention mask
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        # Prepare images batch (list of tuples)
        images_batch = []
        for item in batch_data:
            images_batch.append((item['images_crop'], item['images_ori']))

        # Stack spatial crop info
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim = 0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }

<a name="Train"></a>
### Train the model
This keeps the original notebook training recipe, but uses the Large image preset everywhere, evaluates every 10 steps, and falls back automatically on smaller GPUs.
The custom collator is configured to fail instead of truncating if a sequence would exceed a real tokenizer max length.

We use the custom `DeepSeekOCR2DataCollator` required for DeepSeek-OCR vision finetuning.


In [ ]:
from transformers import Trainer, TrainingArguments
from unsloth import is_bf16_supported

gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024 / 1024 / 1024
if gpu_memory_gb >= 70:
    per_device_train_batch_size = 4
    gradient_accumulation_steps = 2
    per_device_eval_batch_size = 1
elif gpu_memory_gb >= 40:
    per_device_train_batch_size = 2
    gradient_accumulation_steps = 4
    per_device_eval_batch_size = 1
else:
    per_device_train_batch_size = 1
    gradient_accumulation_steps = 8
    per_device_eval_batch_size = 1

torch_dtype_name = "bfloat16" if is_bf16_supported() else "float16"
training_args_payload = {
    "model": "unsloth/DeepSeek-OCR-2",
    "tuner_type": "lora",
    "dataset": TRAIN_DATASET_ID,
    "validation_dataset": VAL_DATASET_ID,
    "use_hf": True,
    "torch_dtype": torch_dtype_name,
    "max_steps": 60,
    "num_train_epochs": None,
    "per_device_train_batch_size": per_device_train_batch_size,
    "per_device_eval_batch_size": per_device_eval_batch_size,
    "learning_rate": 2e-4,
    "lora_rank": 16,
    "lora_alpha": 16,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "output_dir": str(RUN_OUTPUT_DIR),
    "eval_steps": 10,
    "save_steps": 10,
    "save_total_limit": 3,
    "logging_steps": 1,
    "max_length": None,
    "warmup_steps": 5,
    "dataloader_num_workers": 2,
    "eval_on_start": False,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"mode": "unsloth"},
    "truncation_strategy": "disabled_fail_fast",
    "CUDA_VISIBLE_DEVICES": os.environ.get("CUDA_VISIBLE_DEVICES"),
}

print(f"GPU memory: {gpu_memory_gb:.1f} GB")
print(
    f"per_device_train_batch_size={per_device_train_batch_size}, "
    f"gradient_accumulation_steps={gradient_accumulation_steps}, "
    f"per_device_eval_batch_size={per_device_eval_batch_size}, "
    f"effective_batch_size={per_device_train_batch_size * gradient_accumulation_steps}"
)
print(f"Large image preset: base_size={OCR_BASE_SIZE}, image_size={OCR_IMAGE_SIZE}, crop_mode={OCR_CROP_MODE}")
print("No truncation mode: fail fast if a sequence would exceed tokenizer.model_max_length.")

FastVisionModel.for_training(model) # Enable for training!
data_collator = DeepSeekOCR2DataCollator(
    tokenizer = tokenizer,
    model = model,
    image_size = OCR_IMAGE_SIZE,
    base_size = OCR_BASE_SIZE,
    crop_mode = OCR_CROP_MODE,
    train_on_responses_only = True,
    fail_on_sequence_overflow = True,
)

training_args = TrainingArguments(
    per_device_train_batch_size = per_device_train_batch_size,
    per_device_eval_batch_size = per_device_eval_batch_size,
    gradient_accumulation_steps = gradient_accumulation_steps,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    logging_steps = 1,
    eval_strategy = "steps",
    eval_steps = 10,
    save_strategy = "steps",
    save_steps = 10,
    save_total_limit = 3,
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    optim = "adamw_8bit",
    weight_decay = 0.001,
    lr_scheduler_type = "linear",
    seed = 3407,
    fp16 = not is_bf16_supported(),
    bf16 = is_bf16_supported(),
    output_dir = str(RUN_OUTPUT_DIR),
    report_to = "none",
    dataloader_num_workers = 2,
    remove_unused_columns = False,
)

trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = data_collator,
    train_dataset = converted_train_dataset,
    eval_dataset = converted_eval_dataset,
    args = training_args,
)


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

training_log_rows = [row for row in trainer.state.log_history if isinstance(row, dict)]
with BUNDLE_LOGGING_PATH.open("w", encoding = "utf-8") as handle:
    for row in training_log_rows:
        handle.write(json.dumps(row, ensure_ascii = False, default = str) + "\n")

selected_checkpoint = select_best_checkpoint(RUN_OUTPUT_DIR, training_log_rows)
training_args_payload["best_checkpoint_name"] = selected_checkpoint["checkpoint_name"]
training_args_payload["best_checkpoint_path"] = selected_checkpoint["checkpoint_path"]
training_args_payload["train_runtime"] = trainer_stats.metrics.get("train_runtime")
training_args_payload["train_loss"] = trainer_stats.metrics.get("train_loss")
training_args_payload["train_samples"] = len(converted_train_dataset)
training_args_payload["eval_samples"] = len(converted_eval_dataset)
training_args_payload["max_memory_gb"] = max_memory
training_args_payload["peak_reserved_memory_gb"] = used_memory
write_json(BUNDLE_ARGS_PATH, training_args_payload)

print(json.dumps(selected_checkpoint, ensure_ascii = False, indent = 2))
print(f"Saved training log to {BUNDLE_LOGGING_PATH}")


<a name="Inference"></a>
### Inference
Let's run the model!

In [ ]:
prompt = OCR_PROMPT
image_file = "your_image.jpg"
output_path = "your/output/dir"

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 768, image_size = 768, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# This notebook uses the Large preset everywhere.

res = model.infer(tokenizer, prompt = prompt, image_file = image_file,
    output_path = output_path,
    image_size = OCR_IMAGE_SIZE,
    base_size = OCR_BASE_SIZE,
    crop_mode = OCR_CROP_MODE,
    save_results = True,
    test_compress = False)


Compare the finetuned output against `eval_sample[TEXT_COLUMN]` for a quick sanity check on your FineTree sample.


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained(str(FINAL_ADAPTER_DIR))  # Local saving
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
# model.push_to_hub("your_name/deepseek_ocr_lora_finetree_v2", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/deepseek_ocr_lora_finetree_v2", token = "YOUR_HF_TOKEN") # Online saving


Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = str(FINAL_ADAPTER_DIR), # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
        auto_model = AutoModel,
        trust_remote_code = True,
        unsloth_force_compile = True,
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
    )
    FastVisionModel.for_inference(model) # Enable for inference!

prompt = OCR_PROMPT
image_file = "your_image.jpg"
output_path = "your/output/dir"

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 768, image_size = 768, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# This notebook uses the Large preset everywhere.

res = model.infer(tokenizer, prompt = prompt, image_file = image_file,
    output_path = output_path,
    image_size = OCR_IMAGE_SIZE,
    base_size = OCR_BASE_SIZE,
    crop_mode = OCR_CROP_MODE,
    save_results = True,
    test_compress = False)


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit
if False: model.save_pretrained_merged(str(MERGED_16BIT_DIR), tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", tokenizer, token = "YOUR_HF_TOKEN")


<a name="Benchmark"></a>
### Benchmark Evaluation
Run inference on the FineTree validation dataset, materialize `predictions/pred_*.json`, and write a benchmark submission bundle with `info.json`, `args.json`, `logging.jsonl`, and a zip archive.


In [ ]:
def normalize_inference_text(output):
    if isinstance(output, str):
        return output
    if isinstance(output, dict):
        for key in ("text", "response", "prediction", "output"):
            value = output.get(key)
            if isinstance(value, str):
                return value
        return json.dumps(output, ensure_ascii = False)
    return str(output)

def run_deepseek_dataset_inference(model, tokenizer, dataset, raw_predictions_path, output_dir):
    FastVisionModel.for_inference(model)
    raw_predictions_path = Path(raw_predictions_path)
    output_dir = Path(output_dir)
    raw_predictions_path.parent.mkdir(parents = True, exist_ok = True)
    output_dir.mkdir(parents = True, exist_ok = True)

    with raw_predictions_path.open("w", encoding = "utf-8") as handle:
        for prediction_index, sample in enumerate(dataset, start = 1):
            image_path = BENCHMARK_IMAGE_DIR / f"sample_{prediction_index:04d}.png"
            sample[IMAGE_COLUMN].save(image_path)

            response = model.infer(
                tokenizer,
                prompt = OCR_PROMPT,
                image_file = str(image_path),
                output_path = str(BENCHMARK_EVAL_DIR),
                base_size = OCR_BASE_SIZE,
                image_size = OCR_IMAGE_SIZE,
                crop_mode = OCR_CROP_MODE,
                save_results = False,
                test_compress = False,
            )
            response_text = normalize_inference_text(response)
            parsed_prediction = extract_prediction_payload(response_text)
            record = {
                "sample_index": prediction_index,
                "response": response_text,
                "parsed_prediction": parsed_prediction,
                "prediction_file": f"pred_{prediction_index:04d}.json",
                "image_path": image_path.name,
            }
            handle.write(json.dumps(record, ensure_ascii = False) + "\n")
            print(f"[{prediction_index}/{len(dataset)}] wrote pred_{prediction_index:04d}.json")

    return materialize_prediction_json_files(raw_predictions_path, output_dir)


In [ ]:
benchmark_started_utc = utc_now_iso()
benchmark_started_israel = israel_now_iso()
benchmark_started = time.time()
written_predictions = run_deepseek_dataset_inference(
    model,
    tokenizer,
    val_dataset_raw,
    RAW_PREDICTIONS_PATH,
    PREDICTIONS_DIR,
)
benchmark_finished = time.time()
benchmark_finished_utc = utc_now_iso()
benchmark_finished_israel = israel_now_iso()

environment = collect_environment_snapshot()
run_info = {
    "run_id": RUN_ID,
    "workflow_type": "deepseek_ocr_finetune_and_benchmark",
    "train_dataset": TRAIN_DATASET_ID,
    "validation_dataset": VAL_DATASET_ID,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "benchmark_bundle_dir": str(BENCHMARK_BUNDLE_DIR),
    "benchmark_started_at_utc": benchmark_started_utc,
    "benchmark_started_at_israel": benchmark_started_israel,
    "benchmark_finished_at_utc": benchmark_finished_utc,
    "benchmark_finished_at_israel": benchmark_finished_israel,
    "benchmark_duration_seconds": round(benchmark_finished - benchmark_started, 2),
    "prediction_count": len(written_predictions),
}

model_metadata = build_benchmark_model_metadata(
    training_args = training_args_payload,
    environment = environment,
    checkpoint_name = selected_checkpoint["checkpoint_name"],
)

artifacts = {
    "args_json": "args.json",
    "logging_jsonl": "logging.jsonl",
    "prediction_dir": "predictions",
    "raw_predictions_jsonl": "raw_predictions.jsonl",
    "final_adapter_dir": str(FINAL_ADAPTER_DIR),
    "submission_archive": f"{RUN_ARCHIVE_BASE.name}.zip",
}

info_payload = build_submission_info_payload(
    model_metadata = model_metadata,
    training_args = training_args_payload,
    environment = environment,
    run = run_info,
    selected_checkpoint = selected_checkpoint,
    artifacts = artifacts,
)

write_json(BENCHMARK_BUNDLE_DIR / "info.json", info_payload)
(BENCHMARK_BUNDLE_DIR / "README.txt").write_text(
    "This folder is a benchmark submission bundle.\n\n"
    "On the benchmark machine:\n"
    "1. Download and unzip the archive.\n"
    "2. Point benchmark.input_dir at this folder, or copy its contents into the benchmark input folder.\n"
    "3. Make sure the benchmark YAML mappings reference predictions/pred_0001.json, predictions/pred_0002.json, and so on in validation-set order.\n"
    "4. Run finetree-benchmark-run --config benchmark/config.example.yaml --submission /path/to/benchmark_submission\n",
    encoding = "utf-8",
)

archive_path = Path(shutil.make_archive(str(RUN_ARCHIVE_BASE), "zip", root_dir = RUN_OUTPUT_DIR, base_dir = "benchmark_submission"))

print(json.dumps({
    "selected_checkpoint": selected_checkpoint["checkpoint_name"],
    "prediction_files": [path.name for path in written_predictions],
    "bundle_dir": str(BENCHMARK_BUNDLE_DIR),
    "archive_path": str(archive_path),
}, ensure_ascii = False, indent = 2))


The generated zip archive is ready to move to the benchmark repo machine.
Use `info.json` mode there with `finetree-benchmark-run --config benchmark/config.example.yaml --submission /path/to/benchmark_submission`.


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>